In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "heart_disease_clean.csv"
)

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {processed_data_path}"
    )

df = pd.read_csv(processed_data_path)

target_column = "heart_disease"

if target_column not in df.columns:
    raise KeyError(
        f"Target column '{target_column}' was not found."
    )

if df[target_column].isna().any():
    raise ValueError(
        "The target column contains missing values."
    )

X = df.drop(columns=[target_column])
y = df[target_column].astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Full dataset shape:", df.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Training positive rate:", round(y_train.mean() * 100, 2), "%")
print("Testing positive rate:", round(y_test.mean() * 100, 2), "%")

Full dataset shape: (4238, 16)
X_train shape: (3390, 15)
X_test shape: (848, 15)
y_train shape: (3390,)
y_test shape: (848,)
Training positive rate: 15.19 %
Testing positive rate: 15.21 %


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

continuous_features = [
    "age",
    "cigarettes_per_day",
    "total_cholesterol",
    "systolic_bp",
    "diastolic_bp",
    "bmi",
    "heart_rate",
    "glucose"
]

binary_features = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

categorical_features = [
    "gender",
    "education"
]

all_defined_features = (
    continuous_features
    + binary_features
    + categorical_features
)

missing_features = sorted(
    set(X_train.columns) - set(all_defined_features)
)

unexpected_features = sorted(
    set(all_defined_features) - set(X_train.columns)
)

if missing_features:
    raise ValueError(
        f"Features not assigned to a group: {missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"Defined features not found in dataset: {unexpected_features}"
    )

continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            continuous_pipeline,
            continuous_features
        ),
        (
            "binary",
            binary_pipeline,
            binary_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("Tree preprocessing pipeline created successfully")
print("Continuous features:", len(continuous_features))
print("Binary features:", len(binary_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(all_defined_features))

Tree preprocessing pipeline created successfully
Continuous features: 8
Binary features: 5
Categorical features: 2
Total features: 15


In [3]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

decision_tree_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(tree_preprocessor)
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
)

random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(tree_preprocessor)
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

models = {
    "Decision Tree": decision_tree_pipeline,
    "Random Forest": random_forest_pipeline
}

print("Tree model pipelines created successfully")

for model_name in models:
    print("-", model_name)

Tree model pipelines created successfully
- Decision Tree
- Random Forest


In [4]:
import time

training_times = {}

for model_name, model_pipeline in models.items():
    training_start = time.perf_counter()

    model_pipeline.fit(
        X_train,
        y_train
    )

    training_end = time.perf_counter()

    training_times[model_name] = (
        training_end - training_start
    )

    print(
        f"{model_name} trained successfully "
        f"in {training_times[model_name]:.4f} seconds."
    )

training_time_summary = pd.DataFrame({
    "Model": list(training_times.keys()),
    "Training Time (Seconds)": [
        round(value, 4)
        for value in training_times.values()
    ]
})

training_time_summary

Decision Tree trained successfully in 0.0481 seconds.
Random Forest trained successfully in 0.5165 seconds.


,Model,Training Time (Seconds)
0,Decision Tree,0.0481
1,Random Forest,0.5165


In [5]:
train_predictions = {}
test_predictions = {}

train_probabilities = {}
test_probabilities = {}

prediction_times = {}

for model_name, model_pipeline in models.items():
    prediction_start = time.perf_counter()

    train_predictions[model_name] = (
        model_pipeline.predict(X_train)
    )

    test_predictions[model_name] = (
        model_pipeline.predict(X_test)
    )

    train_probabilities[model_name] = (
        model_pipeline.predict_proba(X_train)[:, 1]
    )

    test_probabilities[model_name] = (
        model_pipeline.predict_proba(X_test)[:, 1]
    )

    prediction_end = time.perf_counter()

    prediction_times[model_name] = (
        prediction_end - prediction_start
    )

    print(f"{model_name}:")
    print(
        "  Test predictions:",
        len(test_predictions[model_name])
    )
    print(
        "  Predicted positive cases:",
        int(test_predictions[model_name].sum())
    )
    print(
        "  Probability range:",
        f"{test_probabilities[model_name].min():.4f}",
        "to",
        f"{test_probabilities[model_name].max():.4f}"
    )

prediction_preview = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Decision Tree Prediction":
        test_predictions["Decision Tree"],
    "Decision Tree Probability":
        test_probabilities["Decision Tree"],
    "Random Forest Prediction":
        test_predictions["Random Forest"],
    "Random Forest Probability":
        test_probabilities["Random Forest"]
})

prediction_preview.head(10)

Decision Tree:
  Test predictions: 848
  Predicted positive cases: 153
  Probability range: 0.0000 to 1.0000
Random Forest:
  Test predictions: 848
  Predicted positive cases: 17
  Probability range: 0.0000 to 0.6167


,Actual,Decision Tree Prediction,Decision Tree Probability,Random Forest Prediction,Random Forest Probability
0,0,0,0.0,0,0.350000
1,0,0,0.0,0,0.236667
2,1,0,0.0,0,0.303333
3,0,1,1.0,0,0.413333
4,0,0,0.0,0,0.230000
5,0,1,1.0,0,0.290000
6,0,0,0.0,0,0.223333
7,0,0,0.0,0,0.083333
8,0,0,0.0,0,0.096667
9,0,0,0.0,0,0.230000
